### Скачивание данных

In [ ]:
!mkdir -p data/images

!wget -O /tmp/isic2020_train.zip https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_JPEG.zip
!unzip -q -j /tmp/isic2020_train.zip "*.jpg" -d data/images
!rm /tmp/isic2020_train.zip

!wget -O /tmp/isic2019_train.zip https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip
!unzip -q -j /tmp/isic2019_train.zip "*.jpg" -d data/images
!rm /tmp/isic2019_train.zip

!wget -O /tmp/isic2019_test.zip https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip
!unzip -q -n -j /tmp/isic2019_test.zip "*.jpg" -d data/images
!rm /tmp/isic2019_test.zip

--2026-06-14 21:33:34--  https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_JPEG.zip
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 52.216.51.41, 16.15.191.41, 52.217.167.41, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|52.216.51.41|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 24707698022 (23G) [application/zip]
Saving to: ‘/tmp/isic2020_train.zip’

/tmp/isic2020_train   6%[>                   ]   1.56G  16.9MB/s    eta 21m 47s^C
[/tmp/isic2020_train.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /tmp/isic2020_train.zip or
        /tmp/isic2020_train.zip.zip, and cannot find /tmp/isic2020_train.zip.ZIP, period.
--2026

/tmp/isic2019_train   1%[                    ] 173.72M  16.6MB/s    eta 10m 14s^C
[/tmp/isic2019_train.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /tmp/isic2019_train.zip or
        /tmp/isic2019_train.zip.zip, and cannot find /tmp/isic2019_train.zip.ZIP, period.
--2026-06-14 21:35:24--  https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 16.15.228.160, 54.231.136.201, 16.15.229.226, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|16.15.228.160|:443... ^C
[/tmp/isic2019_test.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a

In [1]:
!wget https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
!wget https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv
!wget https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_GroundTruth.csv

--2026-06-14 21:31:44--  https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 16.15.252.38, 54.231.137.193, 52.216.54.33, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|16.15.252.38|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1291479 (1.2M) [text/csv]
Saving to: ‘ISIC_2019_Training_GroundTruth.csv’

ISIC_2019_Training_ 100%[===================>]   1.23M  1.12MB/s    in 1.1s    

2026-06-14 21:31:46 (1.12 MB/s) - ‘ISIC_2019_Training_GroundTruth.csv’ saved [1291479/1291479]

--2026-06-14 21:31:46--  https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 52.216.209.177, 52.217.126.17, 16.15.213.80, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|52.216.209.177|:443... connected.
HTTP

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import tqdm
import os
from IPython.display import clear_output
from typing import Optional, Dict, List

import torch.utils.data as data
import torchvision.transforms.v2 as v2
import torch
import torch.nn as nn
from torchinfo import summary
from PIL import Image

import wandb
import json
import random

from multiprocessing import Pool
import cv2

sns.set_theme(context="notebook", palette="muted")
RANDOM_STATE = 33

In [3]:
!pip install torchinfo

In [2]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    return seed

In [5]:
def resize_one_cv2(img_path):
    img = cv2.imread(f'data/images/{img_path}')
    if img is None:
        return img_path, 'failed to read'

    img = cv2.resize(img, (224, 224))
    cv2.imwrite(f'data/images/{img_path}', img)

image_paths = os.listdir('data/images')
with Pool(10) as pool:
    list(tqdm.tqdm(pool.imap_unordered(resize_one_cv2, image_paths), total=len(image_paths)))

100%|██████████| 66695/66695 [06:12<00:00, 179.29it/s]


In [3]:
target2019 = pd.read_csv('ISIC_2019_Training_GroundTruth.csv')
target2019test = pd.read_csv('ISIC_2019_Test_GroundTruth.csv')
target2020 = pd.read_csv('ISIC_2020_Training_GroundTruth.csv')

In [4]:
target2020 = target2020[['image_name', 'target']].rename(columns={'image_name' : 'image'})
target2020['target'] = target2020['target'].astype('float64')
target2019 = target2019[['MEL', 'image']].rename(columns={'MEL' : 'target'})
target2019test = target2019test[['MEL', 'image']].rename(columns={'MEL' : 'target'})

In [5]:
labels = pd.concat([target2020, target2019, target2019test], axis=0)

### Класс датасета

In [6]:
class ISICDataset(data.Dataset):
    def __init__(self, root, labels, transforms):
        super().__init__()
        self.transforms = transforms
        self.files = [root + '/' + fname + '.jpg' for fname in labels['image'].tolist()]
        self.targets = labels['target'].tolist()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        fname = self.files[index]
        with Image.open(fname) as im:
            im = im.convert('RGB')
        
        if self.transforms:
            im = self.transforms(im)
        
        return im, self.targets[index]

### Функции для обучения

In [24]:
def training_epoch(model : nn.Module, optimizer : torch.optim.Optimizer, criterion : nn.Module, train_loader : data.DataLoader, device : torch.device, scheduler=None, tqdm_desc='Train'):
    model.train()

    loss_sum = 0.0
    n_samples = 0

    all_probs = []
    all_targets = []

    pbar = tqdm.tqdm(train_loader, desc=tqdm_desc)
    for x, y in pbar:
        x = x.to(device, non_blocking=True).float()
        y = y.to(device, non_blocking=True).float()

        optimizer.zero_grad()

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        if scheduler is not None:
            scheduler.step()

        bs = x.size(0)
        loss_sum += float(loss.item()) * bs
        n_samples += bs

        all_targets.extend(y.cpu().detach().numpy().flatten().tolist())
        all_probs.extend(logits.sigmoid().cpu().detach().numpy().flatten().tolist())

        avg_loss = loss_sum / max(1, n_samples)

        pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'lr': f'{optimizer.param_groups[0]["lr"]:.2e}'})

    return {'loss' : loss_sum / max(1, n_samples), 'roc_auc' : roc_auc_score(all_targets, all_probs)}


@torch.no_grad()
def validation_epoch(model : nn.Module, criterion : nn.Module, valid_loader : data.DataLoader, device : torch.device, tqdm_desc='Valid'):
    model.eval()

    loss_sum = 0.0
    n_samples = 0

    all_probs = []
    all_targets = []

    pbar = tqdm.tqdm(valid_loader, desc=tqdm_desc)
    for x, y in pbar:
        x = x.to(device, non_blocking=True).float()
        y = y.to(device, non_blocking=True).float()

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        bs = x.size(0)
        loss_sum += float(loss.item()) * bs
        n_samples += bs

        all_targets.extend(y.cpu().detach().numpy().flatten().tolist())
        all_probs.extend(logits.sigmoid().cpu().detach().numpy().flatten().tolist())

        avg_loss = loss_sum / max(1, n_samples)

        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    

    return {'loss' : loss_sum / max(1, n_samples), 'roc_auc' : roc_auc_score(all_targets, all_probs)}

def plot_metrics(train_losses, val_losses, train_auc_roc, val_auc_roc):
    clear_output(True)
    epochs = np.arange(1, len(train_losses) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss', marker='o')
    plt.plot(epochs, val_losses, label='Val Loss', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss Dynamics')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_auc_roc, label='Train AUC-ROC', marker='o')
    plt.plot(epochs, val_auc_roc, label='Val AUC-ROC', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('AUC-ROC')
    plt.title('AUC-ROC Dynamics')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.ylim([-0.05, 1.05])
    plt.legend()

    plt.tight_layout()
    plt.show()

def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, val_loss : float) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({'epoch': epoch, 'val_loss': val_loss, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict()}, path)

def fit(model : nn.Module, train_loader : data.DataLoader, val_loader : data.DataLoader, optimizer : torch.optim.Optimizer, scheduler, criterion : nn.Module, device : torch.device, num_epochs: int, out_dir: str = None, plot_fn=None, use_wandb: bool = False, wandb_project: str = None, wandb_run_name: Optional[str] = None, wandb_config: Optional[dict] = None, save_every_n_epochs : int = 1) -> Dict[str, List[float]]:
    model.to(device)

    history = {
        'train_loss': [],
        'val_loss': [],
        'train_roc_auc' : [],
        'val_roc_auc' : [],
        'lr' : [],
    }

    best_roc_auc = 0

    run = None
    if use_wandb:
        run = wandb.init(project=wandb_project, name=wandb_run_name, config=wandb_config if wandb_config else {}, reinit='create_new')
        wandb.config.update({'num_params': sum(p.numel() for p in model.parameters()), 'device': str(device)}, allow_val_change=True)

    for epoch in range(1, num_epochs + 1):
        tr = training_epoch(model=model, train_loader=train_loader, optimizer=optimizer, scheduler=scheduler, criterion=criterion, device=device, tqdm_desc=f'Train {epoch}/{num_epochs}')
        va = validation_epoch(model=model, valid_loader=val_loader, criterion=criterion, device=device, tqdm_desc=f'Valid {epoch}/{num_epochs}')

        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(tr['loss'])
        history['train_roc_auc'].append(tr['roc_auc'])
        history['val_loss'].append(va['loss'])
        history['val_roc_auc'].append(va['roc_auc'])
        history['lr'].append(current_lr)
        
        if epoch % save_every_n_epochs == 0: save_checkpoint(os.path.join(out_dir, 'checkpoints', f"epoch_{epoch:03d}.pt"), model, optimizer, epoch, va['loss'])
        if va['roc_auc'] >= best_roc_auc: 
            save_checkpoint(os.path.join(out_dir, 'checkpoints', 'best.pt'), model, optimizer, epoch, va['loss'])
            best_roc_auc = va['roc_auc']

        print(f'[{epoch:02d}/{num_epochs}] train: loss={tr["loss"]:.4f}, roc_auc={tr["roc_auc"]:.4f} | val: loss={va["loss"]:.4f}, roc_auc={va["roc_auc"]:.4f} | lr={current_lr:.2e}')
        
        if plot_fn:
            plot_fn(history['train_loss'], history['val_loss'], history['train_roc_auc'], history['val_roc_auc'])

        if use_wandb:
            wandb.log({'epoch': epoch, 'train/loss': tr['loss'], 'train/roc_auc': tr['roc_auc'], 'val/loss': va['loss'], 'val/roc_auc': va['roc_auc'], 'lr': current_lr}, step=epoch)

        os.makedirs(f'{out_dir}/logs/', exist_ok=True)
        with open(f'{out_dir}/logs/history.json', 'w', encoding='utf-8') as f:
            json.dump(history, f, ensure_ascii=False, indent=4)

    os.makedirs(os.path.dirname(f'{out_dir}/data/'), exist_ok=True)
    artifact = wandb.Artifact('train_data', type='data')
    with open(f'{out_dir}/data/train_dataset.pkl', 'wb') as f:
        pickle.dump(train_loader.dataset.files, f)
        artifact.add_file(f'{out_dir}/data/train_dataset.pkl', 'train_dataset.pkl')

    wandb.log_artifact(artifact)
    
    artifact = wandb.Artifact('val_data', type='data')
    with open(f'{out_dir}/data/val_dataset.pkl', 'wb') as f:
        pickle.dump(val_loader.dataset.files, f)
        artifact.add_file(f'{out_dir}/data/val_dataset.pkl', 'val_dataset.pkl')

    wandb.log_artifact(artifact)

    artifact = wandb.Artifact('best_model', 'model')
    artifact.add_file(os.path.join(out_dir, 'checkpoints', 'best.pt'), 'best_model.pt')
    
    if use_wandb and run is not None:
        run.finish()

    return history

#### разделим выборку на train, val и test

In [8]:
train_labels, test_labels = train_test_split(labels, test_size=0.3, random_state=RANDOM_STATE, stratify=labels['target'])
test_labels, val_labels = train_test_split(test_labels, test_size=0.35, random_state=RANDOM_STATE, stratify=test_labels['target'])

In [9]:
len(train_labels['target']), len(test_labels['target']), len(val_labels['target'])

(46686, 13005, 7004)

посчитаем статистики по трейн сету

In [10]:
device = torch.device('cuda')

def get_stats(loader):
    csum = torch.zeros(3, device=device)
    csqsum = torch.zeros(3, device=device)
    num_pix = 0

    for b, _ in tqdm.tqdm(loader):
        b = b.to(device)
        csum += (b.sum(dim=[0, 2, 3]))
        csqsum += ((b ** 2).sum(dim=[0, 2, 3]))
        num_pix += (b.shape[0] * b.shape[2] * b.shape[3])
    
    return csum.cpu() / num_pix, torch.sqrt(csqsum.cpu() / num_pix - (csum.cpu() / num_pix) ** 2)

train_mean, train_std = get_stats(data.DataLoader(ISICDataset('data/images', train_labels, transforms=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, True)])), batch_size=128, num_workers=4))

100%|██████████| 365/365 [00:19<00:00, 19.04it/s]


In [11]:
train_mean, train_std

(tensor([0.7280, 0.5716, 0.5542]), tensor([0.2123, 0.2011, 0.2156]))

### Попробуем простую модель

In [26]:
device = torch.device('cuda')

train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=180, interpolation=v2.InterpolationMode.BILINEAR, fill=0),
    v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03),
    v2.RandomApply([v2.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))], p=0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, True),
    v2.Normalize(train_mean, train_std)
])
train_dataset = ISICDataset('data/images', train_labels, transforms=train_transforms)

test_transforms = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, True), v2.Normalize(train_mean, train_std)])

test_dataset = ISICDataset('data/images', test_labels, transforms=test_transforms)
val_dataset = ISICDataset('data/images', val_labels, transforms=test_transforms)

n_pos = train_labels["target"].sum()
n_neg = len(train_labels) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

train_loader = data.DataLoader(train_dataset, batch_size=128, num_workers=8, shuffle=True, persistent_workers=True, prefetch_factor=4)
test_loader = data.DataLoader(test_dataset, batch_size=64)
val_loader = data.DataLoader(val_dataset, batch_size=64, num_workers=2)

model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.GELU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.BatchNorm2d(64),
    nn.GELU(),
    nn.MaxPool2d(2),

    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.GELU(),
    nn.MaxPool2d(2),

    nn.Conv2d(128, 256, kernel_size=3, padding=1),
    nn.BatchNorm2d(256),
    nn.GELU(),
    nn.MaxPool2d(2),

    nn.Conv2d(256, 256, kernel_size=3, padding=1),
    nn.BatchNorm2d(256),
    nn.GELU(),

    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),

    nn.Linear(256, 64),
    nn.GELU(),
    nn.Dropout(0.3),

    nn.Linear(64, 1)
)

num_epochs = 20
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
seed_everything(RANDOM_STATE)

AcceleratorError: CUDA error: invalid argument
Search for `cudaErrorInvalidValue' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [20]:
wandb_config = {
    "task": "melanoma_detection",
    "target": "binary_melanoma_vs_non_melanoma",
    "dataset": "ISIC 2019 + ISIC 2020",
    "metadata": False,
    "image_dir": "data/images",

    "train_size": len(train_labels),
    "val_size": len(val_labels),
    "test_size": len(test_labels),
    "n_pos_train": int(n_pos),
    "n_neg_train": int(n_neg),
    "pos_ratio_train": float(n_pos / len(train_labels)),
    "pos_weight": n_neg / n_pos,

    "image_size": 224,
    "normalization_mean": train_mean.tolist(),
    "normalization_std": train_std.tolist(),

    "augmentations": {
        "random_horizontal_flip": {
            "p": 0.5,
        },
        "random_vertical_flip": {
            "p": 0.5,
        },
        "random_rotation": {
            "degrees": 180,
            "interpolation": "bilinear",
            "fill": 0,
        },
        "color_jitter": {
            "brightness": 0.15,
            "contrast": 0.15,
            "saturation": 0.10,
            "hue": 0.03,
        },
        "gaussian_blur": {
            "p": 0.2,
            "kernel_size": 3,
            "sigma": [0.1, 0.8],
        },
        # "random_erasing": {
        #     "p": 0.3,
        #     "scale": [0.02, 0.05],
        #     "ratio": [0.5, 2.0],
        # },
    },

    "train_batch_size": 128,
    "val_batch_size": 64,
    "test_batch_size": 64,
    "train_num_workers": 8,
    "val_num_workers": 2,
    "pin_memory": True,
    "persistent_workers": True,
    "prefetch_factor": 4,

    "model_name": "SimpleCNN",
    "model_type": "CNN_from_scratch",
    "pretrained": False,
    "activation": "GELU",
    "normalization": "BatchNorm2d",
    "dropout": 0.3,
    "conv_channels": [32, 64, 128, 256, 256],
    "pooling": "MaxPool2d + AdaptiveAvgPool2d",
    "classifier_hidden_dim": 64,
    "output_dim": 1,
    "num_params": sum(p.numel() for p in model.parameters()),
    "num_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),

    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "loss": "BCEWithLogitsLoss",
    "loss_pos_weight": float(pos_weight.detach().cpu().item()),

    "device": str(device),
    "num_epochs": num_epochs,
    "metrics": ["loss", "roc_auc"],
}

In [21]:
fit(model, train_loader, val_loader, optimizer, None, criterion, device, num_epochs, out_dir='simple_cnn', use_wandb=True, wandb_project='gp5_pics_only1', wandb_run_name='gp5_pics_only1_run1', wandb_config=wandb_config, plot_fn=plot_metrics)

Train 1/20:   0%|          | 0/365 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 74.00 MiB. GPU 0 has a total capacity of 22.03 GiB of which 39.69 MiB is free. Process 3477 has 794.00 MiB memory in use. Process 23442 has 13.72 GiB memory in use. Process 47420 has 6.34 GiB memory in use. Including non-PyTorch memory, this process has 1.13 GiB memory in use. Of the allocated memory 865.63 MiB is allocated by PyTorch, and 68.37 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [23]:
validation_epoch(model, criterion, test_loader, device)

Valid: 100%|██████████| 204/204 [00:27<00:00,  7.54it/s, loss=0.8238]


{'loss': 0.8238282012371135, 'roc_auc': np.float64(0.8642751840184085)}

In [24]:
summary(model)

Layer (type:depth-idx)                   Param #
Sequential                               --
├─Conv2d: 1-1                            896
├─BatchNorm2d: 1-2                       64
├─GELU: 1-3                              --
├─MaxPool2d: 1-4                         --
├─Conv2d: 1-5                            18,496
├─BatchNorm2d: 1-6                       128
├─GELU: 1-7                              --
├─MaxPool2d: 1-8                         --
├─Conv2d: 1-9                            73,856
├─BatchNorm2d: 1-10                      256
├─GELU: 1-11                             --
├─MaxPool2d: 1-12                        --
├─Conv2d: 1-13                           295,168
├─BatchNorm2d: 1-14                      512
├─GELU: 1-15                             --
├─MaxPool2d: 1-16                        --
├─Conv2d: 1-17                           590,080
├─BatchNorm2d: 1-18                      512
├─GELU: 1-19                             --
├─AdaptiveAvgPool2d: 1-20                --
├─Fl

In [ ]:
# import math

# num_epochs = 20
# steps_per_epoch = 365

# total_steps = num_epochs * steps_per_epoch
# warmup_steps = steps_per_epoch

# def lr_lambda(step):
#     if step < warmup_steps:
#         return float(step + 1) / float(warmup_steps)

#     progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
#     return 0.5 * (1.0 + math.cos(math.pi * progress))

# scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)